# 当前暂停决定 — 2026-09-26
用户要求当前不做任何筛选，至少等 HCP 3T task 151 人数据完整后再判断。暂停新增筛选、阈值部署及下游重跑。以下单元是历史审计与敏感性分析，当前不要执行。已有task结果未回滚，代码中的门槛未关闭。接手先读根目录 [TSNR_QC_HANDOFF.md](../TSNR_QC_HANDOFF.md)。


# Existing tSNR mask scope audit
2026-09-26. Read-only examination of explicit current branches. Subject-directory population is not necessarily a final analysis cohort. Cortex counts describe represented CIFTI grayordinates, not whole mesh. >=20% is the existing hippocampal QC threshold, not an asserted cortex subject exclusion. Missing paths are reported without substitute selection.


In [1]:
from pathlib import Path
import json, csv, io, contextlib
import numpy as np
ROOT=Path('/home/yojiao/hpchome/hipp')
BRANCHES={
 'stanford36':'outputs_stanford/inputs/stanford/kong2022/shared_inputs',
 'stanford22':'outputs_stanford/inputs/stanford/kong2022_22p/shared_inputs',
 'iowa3t_jj36':'outputs_iowa_jj_tms/inputs/iowa_jj_tms/kong2022/shared_inputs',
 'iowa3t_jj22':'outputs_iowa_jj_tms/inputs/iowa_jj_tms/kong2022/shared_inputs_22p',
 'hcp7rest':'outputs_hcp/inputs/hcp/kong2022/shared_inputs',
 'hcp3emotion':'outputs_hcp/inputs/hcp/kong2022/shared_inputs/task-emotion',
 'hcp3wm':'outputs_hcp/inputs/hcp/kong2022/shared_inputs/task-wm',
}
def safe(p):
 return not any('archive' in x.lower() for x in p.resolve().parts)
def audit():
 for label,rel in BRANCHES.items():
  base=ROOT/rel
  if not base.is_dir(): print(label,'MISSING_ROOT',str(base));continue
  for kind,parent in [('hipp',base/'surface'),('cortex',base/'routea')]:
   if not parent.is_dir(): print(label,kind,'MISSING_PARENT');continue
   subjects=sorted(p for p in parent.iterdir() if p.name.startswith('sub-') and p.is_dir() and safe(p))
   records=[];missing=[];mismatch=[];shapes=set();near_cut=0;nonfinite=0;sdzero=0
   for sub in subjects:
    pair=[]
    for hemi,word in [('L','left'),('R','right')]:
     gate=sub/'surface'/'tsnr_gate' if kind=='hipp' else sub/'reference'/'common'/'tsnr_gate'
     stem=f'{sub.name}_hemi-{hemi}' if kind=='hipp' else f'cortex_{word}'
     p=gate/f'{stem}_valid_mask.npy'
     if not p.is_file() or not safe(p):missing.append(str(p.relative_to(ROOT)));continue
     a=np.load(p,allow_pickle=False)
     if a.dtype!=np.bool_:raise ValueError(f'nonboolean {p}')
     n=a.size;bad=int((~a).sum());shapes.add(tuple(a.shape));pair.append((n,bad))
     records.append((sub.name,hemi,n,bad))
     q=gate/f'{stem}_tsnr.npy'
     if q.is_file() and safe(q):
      t=np.load(q,allow_pickle=False)
      match=np.isfinite(t)&(t>=25)
      if not np.array_equal(a,match):mismatch.append(str(p.relative_to(ROOT)))
      nonfinite+=int((~np.isfinite(t)).sum());near_cut+=int(((t>=20)&(t<=30)).sum())
    if len(pair)==2:pass
   bysub={}
   for s,h,n,b in records:bysub.setdefault(s,[]).append((n,b))
   full=[x for x in bysub.values() if len(x)==2]
   total=sum(x[2] for x in records);bad=sum(x[3] for x in records)
   result={'subject_dirs':len(subjects),'masks':len(records),'complete_subjects':len(full),'total_vertices':total,'invalid_vertices':bad,'invalid_percent':100*bad/total if total else None,'subjects_any_invalid':sum(any(b>0 for n,b in x) for x in full),'subjects_ge20pct_either_hemi':sum(any(b/n>=.2 for n,b in x) for x in full),'all_valid_subjects':sum(all(b==0 for n,b in x) for x in full),'missing_masks':len(missing),'mask_shapes':sorted(shapes),'formula_mismatch':len(mismatch),'nonfinite_tsnr':nonfinite,'tsnr20to30':near_cut}
   print(label,kind,json.dumps(result))
   if missing:print('missing_examples',missing[:3])
   if mismatch:print('mismatch_examples',mismatch[:3])
   affected=[(s,sum(b for n,b in x)) for s,x in bysub.items() if any(b for n,b in x)]
   print('affected_subjects',json.dumps(affected if label not in ['hcp7rest','hcp3emotion','hcp3wm'] else sorted(affected,key=lambda x:-x[1])[:8]))
  qc=base.parent/'qc'/'t1only_hippunfold_tsnr_qc.tsv'
  if qc.is_file() and safe(qc):
   rows=list(csv.DictReader(qc.open(),delimiter='\t'))
   print(label,'qc',len(rows),'fields',list(rows[0]) if rows else [])
   for key in ['pass_tsnr','pass_all','pass_dice']:
    if rows and key in rows[0]:
     vals={str(r[key]) for r in rows};print(key,{v:sum(str(r[key])==v for r in rows) for v in vals})

audit()


stanford36 hipp {"subject_dirs": 196, "masks": 392, "complete_subjects": 196, "total_vertices": 200704, "invalid_vertices": 781, "invalid_percent": 0.38913026147959184, "subjects_any_invalid": 30, "subjects_ge20pct_either_hemi": 1, "all_valid_subjects": 166, "missing_masks": 0, "mask_shapes": [[512]], "formula_mismatch": 0, "nonfinite_tsnr": 781, "tsnr20to30": 0}
affected_subjects [["sub-1009", 5], ["sub-1017", 11], ["sub-1022", 28], ["sub-1038", 3], ["sub-1052", 19], ["sub-1068", 5], ["sub-2011", 16], ["sub-2047", 13], ["sub-2064", 36], ["sub-2083", 8], ["sub-2096", 1], ["sub-2104", 61], ["sub-3022", 19], ["sub-3029", 2], ["sub-3065", 4], ["sub-3086", 21], ["sub-3091", 86], ["sub-3117", 21], ["sub-3120", 39], ["sub-4025", 1], ["sub-4040", 12], ["sub-4044", 1], ["sub-4048", 4], ["sub-4089", 1], ["sub-4096", 6], ["sub-4113", 5], ["sub-4117", 263], ["sub-4135", 5], ["sub-4136", 16], ["sub-4146", 69]]
stanford36 cortex {"subject_dirs": 195, "masks": 390, "complete_subjects": 195, "total_v

In [2]:
from pathlib import Path
import json, io, contextlib, csv
import numpy as np
import nibabel as nib
ROOT=Path('/home/yojiao/hpchome/hipp')
def run():
 for name,rel in [('stanford36','outputs_stanford/inputs/stanford/kong2022/shared_inputs'),('stanford22','outputs_stanford/inputs/stanford/kong2022_22p/shared_inputs'),('iowa3t_jj36','outputs_iowa_jj_tms/inputs/iowa_jj_tms/kong2022/shared_inputs'),('iowa3t_jj22','outputs_iowa_jj_tms/inputs/iowa_jj_tms/kong2022/shared_inputs_22p')]:
  base=ROOT/rel/'surface';counts={'invalid':0,'zero_sd':0,'nonfinite_sd':0,'positive_sd':0,'nonfinite_raw_vertices':0,'mask_raw_mismatch':0};missing=[]
  for sub in sorted(base.iterdir()):
   if not sub.name.startswith('sub-') or not sub.is_dir():continue
   if any('archive' in x.lower() for x in sub.resolve().parts):raise ValueError(str(sub))
   for h in ['L','R']:
    m=sub/'surface'/'tsnr_gate'/f'{sub.name}_hemi-{h}_valid_mask.npy';v=np.load(m)
    if v.all():continue
    p=sub/'surface'/'raw'/f'{sub.name}_hemi-{h}_space-corobl_den-512_label-hipp_bold.func.gii'
    if not p.is_file():missing.append(str(p));continue
    a=np.asarray(nib.load(str(p)).agg_data(),dtype=np.float64)
    if a.shape[0]!=512 and a.shape[1]==512:a=a.T
    sd=np.nanstd(a,axis=1,ddof=1);bad=~v
    counts['invalid']+=int(bad.sum());counts['zero_sd']+=int((sd[bad]==0).sum());counts['nonfinite_sd']+=int((~np.isfinite(sd[bad])).sum());counts['positive_sd']+=int((sd[bad]>0).sum());counts['nonfinite_raw_vertices']+=int((~np.isfinite(a[bad])).any(axis=1).sum());counts['mask_raw_mismatch']+=int(np.sum(v!=(np.isfinite(sd)&(sd>0)&(sd<=400))))
  print(name,'invalid_raw_check',counts,'missing_raw',len(missing),flush=True)
 q=ROOT/'outputs_stanford/inputs/stanford/kong2022/qc/t1only_hippunfold_tsnr_qc.tsv'
 print('Stanford36 excluded QC rows',[r for r in csv.DictReader(q.open(),delimiter='\t') if r['pass_all']!='True'],flush=True)
 base=Path('/home/yojiao/LSS/jianglab/3_Data_Working/fmriprep_processed_Iowa7T')
 for var in ['0P','22P','32P','36P','NoXCPD']:
  root=base/f'DataOrg_PFM_{var}'/'hipp_outputs'/'subjects'
  if not root.is_dir():print('iowa7',var,'missing subjects root',flush=True);continue
  for branch in ['multiband','multi_echo']:
   for kind in ['hipp','cortex']:
    entries=[];missing=0
    for sub in sorted(root.iterdir()):
     if not sub.name.startswith('sub-') or not sub.is_dir():continue
     b=sub/branch
     if not b.is_dir():continue
     for hemi,word in [('L','left'),('R','right')]:
      gate=b/'surface'/'tsnr_gate' if kind=='hipp' else b/'cortex'/'common'/'tsnr_gate'
      stem=f'{sub.name}_hemi-{hemi}' if kind=='hipp' else f'cortex_{word}'
      p=gate/f'{stem}_valid_mask.npy';tp=gate/f'{stem}_tsnr.npy'
      if not p.is_file() or not tp.is_file():missing+=1;continue
      if any('archive' in x.lower() for x in p.resolve().parts):raise ValueError(str(p))
      m=np.load(p);t=np.load(tp);entries.append((sub.name,hemi,int(m.size),int((~m).sum()),int((np.isfinite(t)&~m).sum()),int((~np.isfinite(t)).sum())))
    print('iowa7',var,branch,kind,'masks',len(entries),'missing',missing,'total',sum(x[2] for x in entries),'invalid',sum(x[3] for x in entries),'finite_threshold_reject',sum(x[4] for x in entries),'nonfinite',sum(x[5] for x in entries),'subjects',sorted(set(x[0] for x in entries)),flush=True)

run()


stanford36 invalid_raw_check {'invalid': 781, 'zero_sd': 781, 'nonfinite_sd': 0, 'positive_sd': 0, 'nonfinite_raw_vertices': 0, 'mask_raw_mismatch': 0} missing_raw 0
stanford22 invalid_raw_check {'invalid': 518, 'zero_sd': 518, 'nonfinite_sd': 0, 'positive_sd': 0, 'nonfinite_raw_vertices': 0, 'mask_raw_mismatch': 0} missing_raw 0
iowa3t_jj36 invalid_raw_check {'invalid': 111, 'zero_sd': 111, 'nonfinite_sd': 0, 'positive_sd': 0, 'nonfinite_raw_vertices': 0, 'mask_raw_mismatch': 0} missing_raw 0
iowa3t_jj22 invalid_raw_check {'invalid': 21, 'zero_sd': 21, 'nonfinite_sd': 0, 'positive_sd': 0, 'nonfinite_raw_vertices': 0, 'mask_raw_mismatch': 0} missing_raw 0
Stanford36 excluded QC rows [{'subject': 'sub-1055', 'dice_L': '0.81251175', 'dice_R': '0.63308308', 'hipp_tsnr_invalid_fraction_L': '', 'hipp_tsnr_invalid_fraction_R': '', 'pass_dice': 'False', 'pass_tsnr': 'False', 'pass_all': 'False', 'exclude_reason': 'dice_lt_threshold,hipp_tsnr_invalid_fraction_ge_threshold,missing_tsnr_L,missin

## Denoised-rest follow-up
The following read-only checks were executed as scripts. Their source is retained here for rerunning; measured results and limitations are recorded in `tsnr_cross_dataset_assessment_2026-09-26.md`. Existing result trees were not modified.


In [ ]:
from pathlib import Path
import json,csv
import numpy as np
import nibabel as nib
ROOT=Path('/home/yojiao/hpchome/hipp')
branches={
'stanford36':'outputs_stanford/inputs/stanford/kong2022/shared_inputs',
'stanford22':'outputs_stanford/inputs/stanford/kong2022_22p/shared_inputs',
'jj36':'outputs_iowa_jj_tms/inputs/iowa_jj_tms/kong2022/shared_inputs',
'jj22':'outputs_iowa_jj_tms/inputs/iowa_jj_tms/kong2022/shared_inputs_22p'}
for name,rel in branches.items():
 base=ROOT/rel;changed=0;partial=0;n=0
 for sub in sorted((base/'surface').iterdir()):
  if not sub.name.startswith('sub-') or not sub.is_dir():continue
  for h in ['L','R']:
   raw=sub/f'surface/raw/{sub.name}_hemi-{h}_space-corobl_den-512_label-hipp_bold.func.gii'
   a=np.asarray(nib.load(raw).agg_data(),dtype=float)
   if a.shape[0]!=512:a=a.T
   finite=np.isfinite(a).all(axis=1)
   valid=np.zeros(512,dtype=bool);valid[finite]=np.max(a[finite],axis=1)>np.min(a[finite],axis=1)
   old=np.load(sub/f'surface/tsnr_gate/{sub.name}_hemi-{h}_valid_mask.npy')
   changed+=int((valid!=old).sum());partial+=int((~finite).sum());n+=1
 print(name,'hipp_masks',n,'new_old_differences',changed,'nonfinite_raw_vertices',partial,flush=True)
 if name.startswith('stanford'):
  for sub in sorted((base/'routea').iterdir()):
   if not sub.name.startswith('sub-') or not sub.is_dir():continue
   for h,side in [('L','left'),('R','right')]:
    gate=sub/'reference/common/tsnr_gate'
    t=np.load(gate/f'cortex_{side}_tsnr.npy');old=np.load(gate/f'cortex_{side}_valid_mask.npy')
    idx=np.flatnonzero(np.isfinite(t)&~old)
    if not len(idx):continue
    summary=json.loads((sub/'reference/kong2022/reference_summary.json').read_text())
    print('CORTEX_AFFECTED',name,sub.name,h,'indices',idx.tolist(),'legacy_metric',t[idx].tolist(),'source_fields',{k:v for k,v in summary.items() if k in ['dtseries','dtseries_inputs','route_a_lh_labels','route_a_rh_labels','n_timepoints']},flush=True)


In [ ]:
from pathlib import Path
import csv,json,numpy as np
ROOT=Path('/home/yojiao/hpchome/hipp');J=Path('/home/yojiao/LSS/jianglab/3_Data_Working')
branches=[('stanford36','outputs_stanford/inputs/stanford/kong2022/shared_inputs',J/'fmriprep_processed_CausalConnectome/rest_fmriprep/xcp_d_36P_Head05'),('stanford22','outputs_stanford/inputs/stanford/kong2022_22p/shared_inputs',J/'fmriprep_processed_CausalConnectome/rest_fmriprep/xcpd_zr/xcpd_vol_22P'),('jj36','outputs_iowa_jj_tms/inputs/iowa_jj_tms/kong2022/shared_inputs',J/'fmriprep_processed_JJ_TMS/xcpd/restfmri'),('jj22','outputs_iowa_jj_tms/inputs/iowa_jj_tms/kong2022/shared_inputs_22p',J/'fmriprep_processed_JJ_TMS/xcpd_vol_22P/restfmri')]
for name,rel,source in branches:
 rows=[];missing=[];mismatch=[];subjects=[]
 for sub in sorted((ROOT/rel/'surface').iterdir()):
  if not sub.name.startswith('sub-') or not sub.is_dir():continue
  subjects.append(sub.name)
  files=sorted((source/sub.name/'func').glob(f'{sub.name}_task-rest*_space-MNI152NLin6Asym_desc-linc_qc.tsv'))
  if not files:missing.append(sub.name);continue
  sr=[]
  for p in files:
   data=list(csv.DictReader(p.open(),delimiter='\t'))
   if len(data)!=1:raise ValueError(str(p))
   r=data[0];r['path']=str(p);sr.append(r);rows.append(r)
  ref=ROOT/rel/'routea'/sub.name/'reference/kong2022/reference_summary.json'
  if ref.is_file():
   n=int(np.load(ref.parent/'cortex_roi_parcel_timeseries.npy',mmap_mode='r').shape[1]);ret=sum(int(r['num_retained_volumes']) for r in sr)
   if n!=ret:mismatch.append((sub.name,n,ret))
 if not rows: print(name,'NO_QC',missing,flush=True);continue
 retained=np.array([int(r['num_retained_volumes']) for r in rows]);censored=np.array([int(r['num_censored_volumes']) for r in rows]);fd=np.array([float(r['mean_fd_post_censoring']) for r in rows]);rates=censored/(retained+censored)
 print(name,'subjects',len(subjects),'qc_runs',len(rows),'missing',missing,'retained_frames_min_median_max',[int(retained.min()),float(np.median(retained)),int(retained.max())],'censor_fraction_median_max',[float(np.median(rates)),float(rates.max())],'post_censor_meanFD_median_max',[float(np.median(fd)),float(fd.max())],'frame_mismatch',mismatch,flush=True)
 order=np.argsort(rates)[-5:][::-1]
 for i in order:print('HIGH_CENSOR',name,Path(rows[i]['path']).name,'retained',int(retained[i]),'censored',int(censored[i]),'fraction',round(float(rates[i]),4),'postFD',round(float(fd[i]),4),flush=True)


In [ ]:
from pathlib import Path
import numpy as np,nibabel as nib,json,csv
R=Path('/home/yojiao/hpchome/hipp')
def corr(x,y):
 x=x-x.mean(axis=1,keepdims=True);y=y-y.mean();return (x@y)/(np.linalg.norm(x,axis=1)*np.linalg.norm(y))
for variant,root in [('36p','kong2022'),('22p','kong2022_22p')]:
 base=R/f'outputs_stanford/inputs/stanford/{root}/shared_inputs';ref=base/'routea/sub-1039/reference/kong2022'
 summary=json.loads((ref/'reference_summary.json').read_text());img=nib.load(summary['dtseries']);data=np.asarray(img.dataobj,dtype=np.float64);axis=img.header.get_axis(1)
 rows=list(csv.DictReader((ref/'cortex_roi_parcels.tsv').open(),delimiter='\t'));stored=np.load(ref/'cortex_roi_parcel_timeseries.npy')
 for name,sl,model in axis.iter_structures():
  if name not in ['CIFTI_STRUCTURE_CORTEX_LEFT','CIFTI_STRUCTURE_CORTEX_RIGHT']:continue
  hemi,side,key=('L','left','route_a_lh_labels') if name.endswith('LEFT') else ('R','right','route_a_rh_labels')
  block=data[:,sl].T;gate=ref.parent/'common/tsnr_gate';t=np.load(gate/f'cortex_{side}_tsnr.npy');old=np.load(gate/f'cortex_{side}_valid_mask.npy');changed=np.flatnonzero(np.isfinite(t)&~old)
  if not len(changed):continue
  full=np.loadtxt(summary[key],dtype=int);labels=full[model.vertex];new=old.copy();new[changed]=True
  print(variant,hemi,'vertices',model.vertex[changed].tolist(),'parcels',labels[changed].tolist(),flush=True)
  for pid in sorted(set(labels[changed])):
   if pid==0:continue
   before=block[old&(labels==pid)].mean(axis=0);after=block[new&(labels==pid)].mean(axis=0)
   rowids=[i for i,row in enumerate(rows) if int(row['parcel_id'])==pid]
   assert len(rowids)==1
   np.testing.assert_allclose(before,stored[rowids[0]],rtol=1e-4,atol=1e-4)
   print('ROI',variant,int(pid),'members',int((old&(labels==pid)).sum()),'->',int((new&(labels==pid)).sum()),'old_new_timeseries_r',float(np.corrcoef(before,after)[0,1]),flush=True)
   for smoothing in ['2mm','4mm']:
    for hh in ['L','R']:
     h=np.load(base/f'surface/sub-1039/surface/{smoothing}/sub-1039_hemi-{hh}_timeseries.npy').astype(float)
     delta=abs(corr(h,after)-corr(h,before))
     print('FC_DELTA',variant,int(pid),smoothing,hh,'median',float(np.nanmedian(delta)),'max',float(np.nanmax(delta)),flush=True)


## True tSNR pilot (supersedes finite-validity-only impact conclusions)
The following executed pilot is exploratory: threshold 25 is a comparison value; no production masks were replaced. Rest uses pre-XCP-D samples; task uses raw per-run samples. See the assessment Markdown for limits.

In [ ]:
import sys,json
from pathlib import Path
import numpy as np,nibabel as nib
sys.path.insert(0,str(Path.cwd()/'scripts/core_workflow'))
from lib.workbench import run_workbench
from lib.task_design import build_task_regressors,build_first_level_design,load_nuisance_regressors
from lib.task_glm import fit_first_level_glm
base=Path('/home/yojiao/LSS/jianglab/3_Data_Working')
rows=[]
def metric(path):
    a=np.asarray(nib.load(path).agg_data(),dtype=float)
    return a if a.shape[0]==512 else a.T

def record(dataset,subject,hemi,run,y,design=None):
    mu=y.mean(axis=1)
    detrend=np.column_stack([np.ones(y.shape[1]),np.linspace(-1,1,y.shape[1])])
    noise=y-(detrend@np.linalg.lstsq(detrend,y.T,rcond=None)[0]).T
    for kind,e in [('conventional',y),('linear_detrended',noise)]+([] if design is None else [('task_model_residual',fit_first_level_glm(y,design)[1])]):
        sd=e.std(axis=1,ddof=1)
        valid=np.isfinite(y).all(axis=1)&(mu>0)&(sd>0)
        t=np.full(512,np.nan);t[valid]=mu[valid]/sd[valid]
        row=dict(dataset=dataset,subject=subject,hemi=hemi,run=run,metric=kind,frames=y.shape[1],median_mean=float(np.median(mu)),median_sd=float(np.median(sd)),tsnr_percentiles=np.nanpercentile(t,[5,25,50,75,95]).tolist(),ge25=int(np.sum(t>=25)),valid=int(valid.sum()))
        print(json.dumps(row),flush=True); rows.append(row)
for ds,sub,rel,runprefix in [('stanford','1017','fmriprep_processed_CausalConnectome/rest_fmriprep/fmriprep',''),('iowa_jj_tms','1001','fmriprep_processed_JJ_TMS/derivatives/restfmri','_run-01')]:
    bold=base/rel/f'sub-{sub}/func/sub-{sub}_task-rest{runprefix}_space-MNI152NLin6Asym_res-2_desc-preproc_bold.nii.gz'
    for h in ['L','R']:
        surf=Path(f'outputs_{ds}/inputs/{ds}/hippunfold/dense_corobl_batch/sub-{sub}/hippunfold/sub-{sub}/surf/sub-{sub}_hemi-{h}_space-corobl_den-512_label-hipp_midthickness.surf.gii')
        out=Path(f'/tmp/true_tsnr_{ds}_{sub}_{h}.func.gii')
        run_workbench(['-volume-to-surface-mapping',str(bold),str(surf),str(out),'-trilinear'])
        record(ds,sub,h,runprefix or 'single',metric(out))
for task,n in [('emotion',176),('wm',405)]:
    m=json.loads(Path(f'outputs_hcp/inputs/hcp/kong2022/manifests/task-{task}/sub-115017.json').read_text())
    for h in ['L','R']:
        raw=Path(f'outputs_hcp/inputs/hcp/kong2022/shared_inputs/task-{task}/surface/sub-115017/surface/raw/sub-115017_hemi-{h}_space-corobl_den-512_label-hipp_bold.func.gii')
        y=metric(raw)
        if y.shape[1]!=len(m['runs'])*n: raise ValueError(y.shape)
        for i,r in enumerate(m['runs']):
            rd=Path(r['bold']['path']).parent
            _,ev=build_task_regressors(rd/'EVs',task=task,n_frames=n)
            x=build_first_level_design(ev,load_nuisance_regressors(rd/'Movement_Regressors.txt',n_frames=n))
            record('hcp_'+task,'115017',h,r['run_label'],y[:,i*n:(i+1)*n],x)
Path('/tmp/true_tsnr_pilot_results.json').write_text(json.dumps(rows,indent=2)+'\n')


[
  {
    "dataset": "stanford",
    "subject": "1017",
    "hemi": "L",
    "run": "single",
    "metric": "conventional",
    "frames": 234,
    "median_mean": 3080.33357434395,
    "median_sd": 36.7114502134031,
    "tsnr_percentiles": [
      36.17920403141108,
      52.69979631424749,
      76.56967526737304,
      107.36603501328892,
      153.9197656464211
    ],
    "ge25": 512,
    "valid": 512
  },
  {
    "dataset": "stanford",
    "subject": "1017",
    "hemi": "L",
    "run": "single",
    "metric": "linear_detrended",
    "frames": 234,
    "median_mean": 3080.33357434395,
    "median_sd": 31.09848243106019,
    "tsnr_percentiles": [
      44.81433880906784,
      67.81968157455154,
      92.33302915827709,
      125.8385655047898,
      172.1275544253111
    ],
    "ge25": 512,
    "valid": 512
  },
  {
    "dataset": "stanford",
    "subject": "1017",
    "hemi": "R",
    "run": "single",
    "metric": "conventional",
    "frames": 234,
    "median_mean": 3128.594834026

In [ ]:
import sys,json
from pathlib import Path
import numpy as np,nibabel as nib
sys.path.insert(0,str(Path.cwd()/'scripts/core_workflow'))
from lib.workbench import run_workbench
from lib.task_design import build_task_regressors,build_first_level_design,load_nuisance_regressors
from lib.task_glm import fit_first_level_glm
base=Path('/home/yojiao/LSS/jianglab/3_Data_Working')
rows=[]
def metric(path):
    a=np.asarray(nib.load(path).agg_data(),dtype=float)
    return a if a.shape[0]==512 else a.T

def record(dataset,subject,hemi,run,y,design=None):
    mu=y.mean(axis=1)
    detrend=np.column_stack([np.ones(y.shape[1]),np.linspace(-1,1,y.shape[1])])
    noise=y-(detrend@np.linalg.lstsq(detrend,y.T,rcond=None)[0]).T
    for kind,e in [('conventional',y),('linear_detrended',noise)]+([] if design is None else [('task_model_residual',fit_first_level_glm(y,design)[1])]):
        sd=e.std(axis=1,ddof=1)
        valid=np.isfinite(y).all(axis=1)&(mu>0)&(sd>0)
        t=np.full(512,np.nan);t[valid]=mu[valid]/sd[valid]
        row=dict(dataset=dataset,subject=subject,hemi=hemi,run=run,metric=kind,frames=y.shape[1],median_mean=float(np.median(mu)),median_sd=float(np.median(sd)),tsnr_percentiles=np.nanpercentile(t,[5,25,50,75,95]).tolist(),ge25=int(np.sum(t>=25)),valid=int(valid.sum()))
        print(json.dumps(row),flush=True); rows.append(row)
for ds,sub,rel,runprefix in [('stanford',s,'fmriprep_processed_CausalConnectome/rest_fmriprep/fmriprep','') for s in ['1039','1046','4078']]+[('iowa_jj_tms','1001','fmriprep_processed_JJ_TMS/derivatives/restfmri',f'_run-{r:02d}') for r in [2,3]]:
    bold=base/rel/f'sub-{sub}/func/sub-{sub}_task-rest{runprefix}_space-MNI152NLin6Asym_res-2_desc-preproc_bold.nii.gz'
    for h in ['L','R']:
        surf=Path(f'outputs_{ds}/inputs/{ds}/hippunfold/dense_corobl_batch/sub-{sub}/hippunfold/sub-{sub}/surf/sub-{sub}_hemi-{h}_space-corobl_den-512_label-hipp_midthickness.surf.gii')
        out=Path(f'/tmp/true_tsnr_{ds}_{sub}_{runprefix}_{h}.func.gii')
        run_workbench(['-volume-to-surface-mapping',str(bold),str(surf),str(out),'-trilinear'])
        record(ds,sub,h,runprefix or 'single',metric(out))
Path('/tmp/true_tsnr_rest_followup_results.json').write_text(json.dumps(rows,indent=2)+'\n')


[
  {
    "dataset": "stanford",
    "subject": "1039",
    "hemi": "L",
    "run": "single",
    "metric": "conventional",
    "frames": 234,
    "median_mean": 6255.598865476429,
    "median_sd": 143.87632542645352,
    "tsnr_percentiles": [
      18.575866363163996,
      29.173039836218415,
      43.38151250011892,
      61.96060543938352,
      85.00444478764871
    ],
    "ge25": 426,
    "valid": 512
  },
  {
    "dataset": "stanford",
    "subject": "1039",
    "hemi": "L",
    "run": "single",
    "metric": "linear_detrended",
    "frames": 234,
    "median_mean": 6255.598865476429,
    "median_sd": 132.373755481277,
    "tsnr_percentiles": [
      20.233258204486305,
      31.6098400604154,
      47.6151293990586,
      68.29890645502526,
      93.7431059212862
    ],
    "ge25": 450,
    "valid": 512
  },
  {
    "dataset": "stanford",
    "subject": "1039",
    "hemi": "R",
    "run": "single",
    "metric": "conventional",
    "frames": 234,
    "median_mean": 6885.5067983

## Actual runwise tSNR implementation and task deployment
The cells below preserve executed analysis source and its actual JSON results. Production task 2/4mm products were rebuilt; rest comparisons do not overwrite formal products.

In [ ]:
import sys,json
from pathlib import Path
import numpy as np,nibabel as nib
sys.path.insert(0,str(Path.cwd()/'scripts/core_workflow'))
from lib.tsnr import task_tsnr
rows=[]
additional={'162935','164636','165436','167036','167440','169343','169444','169747','171633'}
for task in ['emotion','wm']:
    base=Path(f'outputs_hcp/inputs/hcp/kong2022/shared_inputs/task-{task}')
    group=json.loads(Path(f'outputs_hcp/work/task_activation/task-{task}/group/group_build_manifest.json').read_text())
    for s0 in group['roster']:
        s=s0.removeprefix('sub-');root=base if s in additional else base/'surface'
        d=root/f'sub-{s}/surface'
        sampling=json.loads((d/'raw/surface_sampling_summary.json').read_text())
        for h in ['L','R']:
            a=np.asarray(nib.load(str(d/f'raw/sub-{s}_hemi-{h}_space-corobl_den-512_label-hipp_bold.func.gii')).agg_data(),dtype=float)
            if a.shape[0]!=512:a=a.T
            scores,new=task_tsnr(a,sampling['run_lengths'],sampling['bold_inputs'],task=task)
            old=10000/a.std(axis=1,ddof=1)>=25
            current=np.load(d/f'tsnr_gate/sub-{s}_hemi-{h}_valid_mask.npy')
            row=dict(task=task,subject=s,hemi=h,original_valid=int(old.sum()),current_valid=int(current.sum()),new_valid=int(new.sum()),rescued=int((new&~old).sum()),newly_excluded=int((old&~new).sum()),run_valid=(scores>=25).sum(axis=1).tolist(),invalid_fraction=float(1-new.mean()))
            rows.append(row)
            np.savez(f'/tmp/task_tsnr_{task}_{s}_{h}.npz',scores=scores,valid=new)
        print(task,s,rows[-2]['new_valid'],rows[-1]['new_valid'],flush=True)
Path('/tmp/task_tsnr_change_results.json').write_text(json.dumps(rows,indent=2))
for t in ['emotion','wm']:
    rr=[r for r in rows if r['task']==t]
    print(t,{k:sum(r[k] for r in rr) for k in ['original_valid','current_valid','new_valid','rescued','newly_excluded']},'subjects_ge20pct_invalid',len({r['subject'] for r in rr if r['invalid_fraction']>=.2}))


[
  {
    "task": "emotion",
    "subject": "114823",
    "hemi": "L",
    "original_valid": 189,
    "current_valid": 512,
    "new_valid": 432,
    "rescued": 247,
    "newly_excluded": 4,
    "run_valid": [
      455,
      449
    ],
    "invalid_fraction": 0.15625
  },
  {
    "task": "emotion",
    "subject": "114823",
    "hemi": "R",
    "original_valid": 115,
    "current_valid": 512,
    "new_valid": 423,
    "rescued": 314,
    "newly_excluded": 6,
    "run_valid": [
      441,
      445
    ],
    "invalid_fraction": 0.173828125
  },
  {
    "task": "emotion",
    "subject": "115017",
    "hemi": "L",
    "original_valid": 46,
    "current_valid": 512,
    "new_valid": 286,
    "rescued": 242,
    "newly_excluded": 2,
    "run_valid": [
      303,
      371
    ],
    "invalid_fraction": 0.44140625
  },
  {
    "task": "emotion",
    "subject": "115017",
    "hemi": "R",
    "original_valid": 91,
    "current_valid": 512,
    "new_valid": 326,
    "rescued": 240,
    "newly

In [ ]:
import sys,json
from pathlib import Path
import numpy as np,nibabel as nib
sys.path.insert(0,str(Path.cwd()/'scripts/core_workflow'))
from lib.io_utils import compute_cortex_tsnr_gate
from lib.extract_cortex_merged_network_timeseries import extract_structure_data
rows=[]
for task in ['emotion','wm']:
    records=[r for r in json.loads(Path('/tmp/task_tsnr_change_results.json').read_text()) if r['task']==task]
    for sub in sorted({r['subject'] for r in records}):
        manifest=json.loads(Path(f'outputs_hcp/inputs/hcp/kong2022/manifests/task-{task}/sub-{sub}.json').read_text())
        parts=[];lengths=[];bolds=[];axis=None
        for r in manifest['runs']:
            image=nib.load(r['dtseries']['path']);a=np.asarray(image.dataobj,dtype=np.float32)
            newaxis=image.header.get_axis(1)
            if axis is not None and axis!=newaxis:raise ValueError('Brain model mismatch')
            axis=newaxis;parts.append(a);lengths.append(a.shape[0]);bolds.append(r['bold']['path'])
        a=np.concatenate(parts,axis=0).T;del parts
        for h,structure in [('left','CORTEX_LEFT'),('right','CORTEX_RIGHT')]:
            ids,y=extract_structure_data(axis,a,structure)
            original=np.isfinite(y).all(axis=1)&(y.std(axis=1,ddof=1)>0)&(10000/np.maximum(y.std(axis=1,ddof=1),1e-30)>=25)
            out=Path(f'/tmp/tsnr_integrated/task-{task}/cortex/sub-{sub}')
            p=compute_cortex_tsnr_gate(data=y,outdir=out,hemi=h,threshold=25,task=task,run_lengths=lengths,task_bold_paths=bolds)
            new=np.load(p)
            rows.append(dict(task=task,subject=sub,hemi=h,total=len(new),old_valid=int(original.sum()),new_valid=int(new.sum()),rescued=int((new&~original).sum()),newly_excluded=int((original&~new).sum())))
        print(task,sub,'cortex completed',flush=True)
Path('/tmp/task_cortex_tsnr_change_results.json').write_text(json.dumps(rows,indent=2))
for task in ['emotion','wm']:
    rr=[r for r in rows if r['task']==task];print(task,{k:sum(r[k] for r in rr) for k in ['total','old_valid','new_valid','rescued','newly_excluded']})


[
  {
    "task": "emotion",
    "subject": "114823",
    "hemi": "left",
    "total": 29696,
    "old_valid": 23100,
    "new_valid": 28930,
    "rescued": 5916,
    "newly_excluded": 86
  },
  {
    "task": "emotion",
    "subject": "114823",
    "hemi": "right",
    "total": 29716,
    "old_valid": 22623,
    "new_valid": 28797,
    "rescued": 6279,
    "newly_excluded": 105
  },
  {
    "task": "emotion",
    "subject": "115017",
    "hemi": "left",
    "total": 29696,
    "old_valid": 19560,
    "new_valid": 28619,
    "rescued": 9138,
    "newly_excluded": 79
  },
  {
    "task": "emotion",
    "subject": "115017",
    "hemi": "right",
    "total": 29716,
    "old_valid": 20433,
    "new_valid": 28768,
    "rescued": 8423,
    "newly_excluded": 88
  },
  {
    "task": "emotion",
    "subject": "115825",
    "hemi": "left",
    "total": 29696,
    "old_valid": 22577,
    "new_valid": 28731,
    "rescued": 6239,
    "newly_excluded": 85
  },
  {
    "task": "emotion",
    "subject"

In [ ]:
import csv,json,sys
from pathlib import Path
import numpy as np,nibabel as nib
sys.path.insert(0,str(Path.cwd()/'scripts/core_workflow'))
from lib.tsnr import rest_tsnr_input
base=Path('/home/yojiao/LSS/jianglab/3_Data_Working/fmriprep_processed_CausalConnectome/rest_fmriprep')
rows=[]
for branch,qcdir,root in [('36p',base/'xcp_d_36P_Head05',Path('outputs_stanford/inputs/stanford/kong2022/shared_inputs')),('22p',base/'xcpd_zr/xcpd_vol_22P',Path('outputs_stanford/inputs/stanford/kong2022_22p/shared_inputs'))]:
    for sub in (['1017','1039','1046','4078'] if branch=='36p' else ['1017','1039','4078']):
        prefix=f'sub-{sub}_task-rest'
        qc=next(iter(csv.DictReader((qcdir/f'sub-{sub}/func/{prefix}_space-MNI152NLin6Asym_desc-linc_qc.tsv').open(),delimiter='\t')))
        outliers=list(csv.DictReader((qcdir/f'sub-{sub}/func/{prefix}_outliers.tsv').open(),delimiter='\t'))
        dummy=int(qc['num_dummy_volumes'])
        keep=np.array([i+dummy for i,r in enumerate(outliers) if float(r['framewise_displacement'])==0],dtype=int)
        if len(keep)!=int(qc['num_retained_volumes']):raise ValueError((sub,len(keep),qc))
        for h in ['L','R']:
            sampled=Path(f'/tmp/true_tsnr_stanford_{sub}_{h}.func.gii') if sub=='1017' else Path(f'/tmp/true_tsnr_stanford_{sub}__{h}.func.gii')
            y=np.asarray(nib.load(str(sampled)).agg_data(),dtype=float)
            if y.shape[0]!=512:y=y.T
            if len(outliers)+dummy!=y.shape[1]:raise ValueError('outlier/original-frame mismatch')
            p=Path(f'/tmp/tsnr_stanford_{branch}_{sub}_{h}.npz')
            np.savez(p,dataset='stanford',subject=sub,hemi=h,n_runs=1,run_0=y,frames_0=keep)
            scores,new=rest_tsnr_input(p,dataset='stanford',subject=sub,hemi=h,n_vertices=512)
            old=np.load(root/f'surface/sub-{sub}/surface/tsnr_gate/sub-{sub}_hemi-{h}_valid_mask.npy')
            # Analysis residual validity stays a prerequisite.
            new &= old  # In these hippocampal roots old exclusions are exactly zero-variance.
            rows.append(dict(dataset='stanford',branch=branch,subject=sub,hemi=h,frames=len(keep),old_valid=int(old.sum()),new_valid=int(new.sum()),changed=int(np.sum(old!=new))))
            print(rows[-1],flush=True)
Path('/tmp/rest_tsnr_retained_results.json').write_text(json.dumps(rows,indent=2))


[
  {
    "dataset": "stanford",
    "branch": "36p",
    "subject": "1017",
    "hemi": "L",
    "frames": 231,
    "old_valid": 512,
    "new_valid": 512,
    "changed": 0
  },
  {
    "dataset": "stanford",
    "branch": "36p",
    "subject": "1017",
    "hemi": "R",
    "frames": 231,
    "old_valid": 501,
    "new_valid": 501,
    "changed": 0
  },
  {
    "dataset": "stanford",
    "branch": "36p",
    "subject": "1039",
    "hemi": "L",
    "frames": 215,
    "old_valid": 512,
    "new_valid": 465,
    "changed": 47
  },
  {
    "dataset": "stanford",
    "branch": "36p",
    "subject": "1039",
    "hemi": "R",
    "frames": 215,
    "old_valid": 512,
    "new_valid": 428,
    "changed": 84
  },
  {
    "dataset": "stanford",
    "branch": "36p",
    "subject": "1046",
    "hemi": "L",
    "frames": 67,
    "old_valid": 512,
    "new_valid": 316,
    "changed": 196
  },
  {
    "dataset": "stanford",
    "branch": "36p",
    "subject": "1046",
    "hemi": "R",
    "frames": 67,


### Preliminary JJ_TMS 1001 pilot (superseded for cohort impact)

This exploratory cell used the 36P surface and enclosing mapping for both variant masks. Its JJ22 rows are superseded by the corrected full-cohort trilinear calculation in the later cell; use the corrected table in the assessment. The JJ36 rows remain the matching pilot.


In [ ]:
import csv,json,sys
from pathlib import Path
import numpy as np,nibabel as nib
sys.path.insert(0,str(Path.cwd()/'scripts/core_workflow'))
from lib.workbench import run_workbench
from lib.tsnr import rest_tsnr_input
base=Path('/home/yojiao/LSS/jianglab/3_Data_Working/fmriprep_processed_JJ_TMS')
sub='1001';samples={h:[] for h in ['L','R']}
for r in [1,2,3]:
    prefix=f'sub-{sub}_task-rest_run-{r:02d}'
    bold=base/f'derivatives/restfmri/sub-{sub}/func/{prefix}_space-MNI152NLin6Asym_res-2_desc-preproc_bold.nii.gz'
    for h in ['L','R']:
        surf=Path(f'outputs_iowa_jj_tms/inputs/iowa_jj_tms/hippunfold/dense_corobl_batch/sub-{sub}/hippunfold/sub-{sub}/surf/sub-{sub}_hemi-{h}_space-corobl_den-512_label-hipp_midthickness.surf.gii')
        out=Path(f'/tmp/jj_true_tsnr_{sub}_{r}_{h}.func.gii')
        run_workbench(['-volume-to-surface-mapping',str(bold),str(surf),str(out),'-enclosing'])
        y=np.asarray(nib.load(str(out)).agg_data(),dtype=float)
        samples[h].append(y if y.shape[0]==512 else y.T)
    print('sampled',r,flush=True)
rows=[]
for branch,qcdir,root in [('36p',base/'xcpd/restfmri',Path('outputs_iowa_jj_tms/inputs/iowa_jj_tms/kong2022/shared_inputs')),('22p',base/'xcpd_vol_22P/restfmri',Path('outputs_iowa_jj_tms/inputs/iowa_jj_tms/kong2022/shared_inputs_22p'))]:
    frames=[]
    for r in [1,2,3]:
        prefix=f'sub-{sub}_task-rest_run-{r:02d}'
        qc=list(csv.DictReader((qcdir/f'sub-{sub}/func/{prefix}_space-MNI152NLin6Asym_desc-linc_qc.tsv').open(),delimiter='\t'))[0]
        mask=list(csv.DictReader((qcdir/f'sub-{sub}/func/{prefix}_outliers.tsv').open(),delimiter='\t'))
        dummy=int(qc['num_dummy_volumes']);keep=np.array([i+dummy for i,row in enumerate(mask) if float(row['framewise_displacement'])==0],dtype=int)
        if len(keep)!=int(qc['num_retained_volumes']) or len(mask)+dummy!=230:raise ValueError('frame matching error')
        frames.append(keep)
    for h in ['L','R']:
        p=Path(f'/tmp/tsnr_jj_{branch}_{sub}_{h}.npz')
        fields=dict(dataset='iowa_jj_tms',subject=sub,hemi=h,n_runs=3)
        for i in range(3):fields.update({f'run_{i}':samples[h][i],f'frames_{i}':frames[i]})
        np.savez(p,**fields)
        scores,new=rest_tsnr_input(p,dataset='iowa_jj_tms',subject=sub,hemi=h,n_vertices=512)
        old=np.load(root/f'surface/sub-{sub}/surface/tsnr_gate/sub-{sub}_hemi-{h}_valid_mask.npy');new &= old
        rows.append(dict(dataset='iowa_jj_tms',branch=branch,subject=sub,hemi=h,old_valid=int(old.sum()),new_valid=int(new.sum()),changed=int((old!=new).sum()),run_pass=(scores>=25).sum(axis=1).tolist(),retained=[len(x) for x in frames]))
        print(rows[-1],flush=True)
Path('/tmp/jj_tsnr_retained_results.json').write_text(json.dumps(rows,indent=2))


[
  {
    "dataset": "iowa_jj_tms",
    "branch": "36p",
    "subject": "1001",
    "hemi": "L",
    "old_valid": 512,
    "new_valid": 501,
    "changed": 11,
    "run_pass": [
      509,
      505,
      507
    ],
    "retained": [
      230,
      228,
      228
    ]
  },
  {
    "dataset": "iowa_jj_tms",
    "branch": "36p",
    "subject": "1001",
    "hemi": "R",
    "old_valid": 512,
    "new_valid": 482,
    "changed": 30,
    "run_pass": [
      510,
      486,
      495
    ],
    "retained": [
      230,
      228,
      228
    ]
  },
  {
    "dataset": "iowa_jj_tms",
    "branch": "22p",
    "subject": "1001",
    "hemi": "L",
    "old_valid": 512,
    "new_valid": 501,
    "changed": 11,
    "run_pass": [
      509,
      505,
      507
    ],
    "retained": [
      230,
      228,
      228
    ]
  },
  {
    "dataset": "iowa_jj_tms",
    "branch": "22p",
    "subject": "1001",
    "hemi": "R",
    "old_valid": 512,
    "new_valid": 482,
    "changed": 30,
    "run_pas

In [ ]:
import csv,json,sys
from pathlib import Path
import numpy as np,nibabel as nib
sys.path.insert(0,str(Path.cwd()/'scripts/core_workflow'))
from lib.tsnr import rest_tsnr_input
from lib.workbench import run_workbench
base=Path('/home/yojiao/LSS/jianglab/3_Data_Working/fmriprep_processed_Iowa7T');sub='06112026'
prod=base/f'DataOrg_PFM_36P/hipp_outputs/subjects/sub-{sub}'
rows=[]
for branch,source,qcroot,stems in [('multiband',base/'derivatives',base/'xcpd/xcpd_vol_36P',['dir-AP_run-04','dir-PA_run-03']),('multi_echo',base/'xcpd/tedana_input',base/'xcpd/tedana_output_36P',['run-01','run-02'])]:
    samples={h:[] for h in ['L','R']};frames=[]
    for run,stem in enumerate(stems):
        prefix=f'sub-{sub}_task-rest_{stem}';bold=source/f'sub-{sub}/func/{prefix}_space-MNI152NLin6Asym_res-2_desc-preproc_bold.nii.gz'
        qc=list(csv.DictReader((qcroot/f'sub-{sub}/func/{prefix}_space-MNI152NLin6Asym_desc-linc_qc.tsv').open(),delimiter='\t'))[0]
        mask=list(csv.DictReader((qcroot/f'sub-{sub}/func/{prefix}_outliers.tsv').open(),delimiter='\t'))
        dummy=int(qc['num_dummy_volumes']);keep=np.array([i+dummy for i,r in enumerate(mask) if float(r['framewise_displacement'])==0],dtype=int)
        if len(keep)!=int(qc['num_retained_volumes']):raise ValueError('Retained frame count mismatch')
        frames.append(keep)
        for h in ['L','R']:
            surf=prod/f'hippunfold/sub-{sub}/surf/sub-{sub}_hemi-{h}_space-corobl_den-512_label-hipp_midthickness.surf.gii'
            out=Path(f'/tmp/iowa7_tsnr_{sub}_{stem}_{h}.func.gii')
            run_workbench(['-volume-to-surface-mapping',str(bold),str(surf),str(out),'-trilinear'])
            y=np.asarray(nib.load(str(out)).agg_data(),dtype=float);y=y if y.shape[0]==512 else y.T
            if len(mask)+dummy!=y.shape[1]:raise ValueError('Original frame count mismatch')
            samples[h].append(y)
        print('Sampled',branch,stem,flush=True)
    for h in ['L','R']:
        fields=dict(dataset='iowa7t',subject=sub,hemi=h,n_runs=2)
        for i in range(2):fields.update({f'run_{i}':samples[h][i],f'frames_{i}':frames[i]})
        p=Path(f'/tmp/tsnr_iowa7t_36p_{sub}_{branch}_{h}.npz');np.savez(p,**fields)
        scores,new=rest_tsnr_input(p,dataset='iowa7t',subject=sub,hemi=h,n_vertices=512)
        old=np.load(prod/f'{branch}/surface/tsnr_gate/sub-{sub}_hemi-{h}_valid_mask.npy');new &= old
        rows.append(dict(dataset='iowa7t',variant='36p',subject=sub,branch=branch,hemi=h,old_valid=int(old.sum()),new_valid=int(new.sum()),changed=int((old!=new).sum()),retained=[len(k) for k in frames],median_baseline=[float(y[:,k].mean(axis=1).mean()) for y,k in zip(samples[h],frames)]))
        print(rows[-1],flush=True)
Path('/tmp/iowa7t_tsnr_retained_results.json').write_text(json.dumps(rows,indent=2))


[
  {
    "dataset": "iowa7t",
    "variant": "36p",
    "subject": "06112026",
    "branch": "multiband",
    "hemi": "L",
    "old_valid": 512,
    "new_valid": 319,
    "changed": 193,
    "retained": [
      386,
      377
    ],
    "median_baseline": [
      1402.9419774592232,
      1335.9042984785349
    ]
  },
  {
    "dataset": "iowa7t",
    "variant": "36p",
    "subject": "06112026",
    "branch": "multiband",
    "hemi": "R",
    "old_valid": 512,
    "new_valid": 334,
    "changed": 178,
    "retained": [
      386,
      377
    ],
    "median_baseline": [
      1385.4979194217394,
      1231.0564829906357
    ]
  },
  {
    "dataset": "iowa7t",
    "variant": "36p",
    "subject": "06112026",
    "branch": "multi_echo",
    "hemi": "L",
    "old_valid": 512,
    "new_valid": 501,
    "changed": 11,
    "retained": [
      399,
      392
    ],
    "median_baseline": [
      915.9698278760504,
      921.0325642170656
    ]
  },
  {
    "dataset": "iowa7t",
    "variant":

## Full Stanford/JJ rest impact computation

The source cell below is the executed full-cohort computation. It uses the actual sampling method per branch: Stanford and JJ_TMS 22P use trilinear mapping; JJ_TMS 36P uses enclosing-voxel mapping. The attached result combines the full-cohort run with the separately executed JJ_TMS 22P hippocampal trilinear rerun (`/tmp/jj22_trilinear_tsnr_results.json`). JJ22 cortex rows and the other branches are from the original full-cohort run. The Stanford sub-4117 hippocampal supplement is in the following cells; its absent cortex reference remains an error.


In [ ]:
import os,sys,json,csv,traceback
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor,as_completed
import numpy as np,nibabel as nib
sys.path.insert(0,str(Path.cwd()/'scripts/core_workflow'))
from lib.tsnr import runwise_tsnr
from lib.task_quality import rest_signal_validity
from lib.workbench import run_workbench
from lib.extract_cortex_merged_network_timeseries import extract_structure_data
J=Path('/home/yojiao/LSS/jianglab/3_Data_Working');OUT=Path('/tmp/full_rest_tsnr');OUT.mkdir(exist_ok=True)
configs={
'stanford36':dict(ds='stanford',root=Path('outputs_stanford/inputs/stanford/kong2022/shared_inputs'),qc=J/'fmriprep_processed_CausalConnectome/rest_fmriprep/xcp_d_36P_Head05'),
'stanford22':dict(ds='stanford',root=Path('outputs_stanford/inputs/stanford/kong2022_22p/shared_inputs'),qc=J/'fmriprep_processed_CausalConnectome/rest_fmriprep/xcpd_zr/xcpd_vol_22P'),
'jj36':dict(ds='iowa_jj_tms',root=Path('outputs_iowa_jj_tms/inputs/iowa_jj_tms/kong2022/shared_inputs'),qc=J/'fmriprep_processed_JJ_TMS/xcpd/restfmri'),
'jj22':dict(ds='iowa_jj_tms',root=Path('outputs_iowa_jj_tms/inputs/iowa_jj_tms/kong2022/shared_inputs_22p'),qc=J/'fmriprep_processed_JJ_TMS/xcpd_vol_22P/restfmri')}
jobs={}
for branch,c in configs.items():
 for d in (c['root']/'surface').iterdir():
  if d.is_dir() and d.name.startswith('sub-'):
   jobs.setdefault((c['ds'],d.name.removeprefix('sub-')),[]).append(branch)
def calculate(key,branches):
 ds,sub=key;dest=OUT/ds/sub;dest.mkdir(parents=True,exist_ok=True)
 info={};ids=set()
 for branch in branches:
  c=configs[branch];ref=c['root']/f'routea/sub-{sub}/reference/kong2022/reference_summary.json'
  if branch=='jj36':
   m=json.loads(Path(f'outputs_iowa_jj_tms/work/hcp151_fixed_template_matching/subjects/sub-{sub}/subject_manifest.json').read_text())
   runs=[r['run_id'] for r in m['header_evidence']['runs']]
   paths=[Path(m['inputs'][f'run-{r:02d}_dtseries']['path']) for r in runs]
  else:
   m=json.loads(ref.read_text())
   if ds=='stanford':runs=[0];paths=[Path(m['dtseries'])]
   else:runs=m['run_ids'];paths=[Path(p) for p in m['dtseries_inputs']]
  info[branch]=(runs,paths);ids.update(runs)
 surfaces=[nib.load(f'outputs_{ds}/inputs/{ds}/hippunfold/dense_corobl_batch/sub-{sub}/hippunfold/sub-{sub}/surf/sub-{sub}_hemi-{h}_space-corobl_den-512_label-hipp_midthickness.surf.gii') for h in ['L','R']]
 xyz=np.concatenate([s.get_arrays_from_intent('NIFTI_INTENT_POINTSET')[0].data for s in surfaces]);tri=np.concatenate([s.get_arrays_from_intent('NIFTI_INTENT_TRIANGLE')[0].data+i*512 for i,s in enumerate(surfaces)])
 surf=dest/'bilateral.surf.gii';nib.save(nib.gifti.GiftiImage(darrays=[nib.gifti.GiftiDataArray(xyz.astype('float32'),intent='NIFTI_INTENT_POINTSET'),nib.gifti.GiftiDataArray(tri.astype('int32'),intent='NIFTI_INTENT_TRIANGLE')]),surf)
 hippocampus={};cortex={};vertex_ids={}
 for r in sorted(ids):
  stem=f'sub-{sub}_task-rest'+(f'_run-{r:02d}' if r else '')
  if ds=='stanford':b=J/f'fmriprep_processed_CausalConnectome/rest_fmriprep/fmriprep/sub-{sub}/func';cb=J/f'fmriprep_processed_CausalConnectome/rest_fmriprep/derivatives_surf/sub-{sub}/func'
  else:b=J/f'fmriprep_processed_JJ_TMS/derivatives/restfmri/sub-{sub}/func';cb=J/f'fmriprep_processed_JJ_TMS/derivatives_surf/sub-{sub}/func'
  volume=b/f'{stem}_space-MNI152NLin6Asym_res-2_desc-preproc_bold.nii.gz'
  cifti=cb/f'{stem}_space-fsLR_den-91k_bold.dtseries.nii'
  if not volume.is_file() or not cifti.is_file():raise FileNotFoundError(f'Explicit baseline files: {volume}; {cifti}')
  sample=dest/f'run-{r}.func.gii';run_workbench(['-volume-to-surface-mapping',str(volume),str(surf),str(sample),'-trilinear' if ds=='stanford' or branch=='jj22' else '-enclosing'])
  values=np.column_stack([a.data for a in nib.load(sample).darrays]);hippocampus[r]={'L':values[:512], 'R':values[512:]}
  im=nib.load(cifti);dt=np.asarray(im.dataobj,dtype=np.float32).T;axis=im.header.get_axis(1);cortex[r]={}
  for h,structure in [('L','CORTEX_LEFT'),('R','CORTEX_RIGHT')]:
   v,y=extract_structure_data(axis,dt,structure)
   if h in vertex_ids and not np.array_equal(v,vertex_ids[h]):raise ValueError('Baseline CIFTI run vertex order differs')
   vertex_ids[h]=v;cortex[r][h]=y
 rows=[]
 for branch,(runs,paths) in info.items():
  c=configs[branch];frames=[]
  for r in runs:
   stem=f'sub-{sub}_task-rest'+(f'_run-{r:02d}' if r else '')
   q=c['qc']/f'sub-{sub}/func';qc=list(csv.DictReader((q/f'{stem}_space-MNI152NLin6Asym_desc-linc_qc.tsv').open(),delimiter='\t'))[0]
   mask=list(csv.DictReader((q/f'{stem}_outliers.tsv').open(),delimiter='\t'));dummy=int(qc['num_dummy_volumes'])
   keep=np.array([i+dummy for i,row in enumerate(mask) if float(row['framewise_displacement'])==0],dtype=int)
   if len(keep)!=int(qc['num_retained_volumes']) or len(mask)+dummy!=hippocampus[r]['L'].shape[1]:raise ValueError('Original/retained frame mismatch')
   if cortex[r]['L'].shape[1]!=hippocampus[r]['L'].shape[1]:raise ValueError('Baseline volume/CIFTI time mismatch')
   frames.append(keep)
  den={h:[] for h in ['L','R']}
  for p in paths:
   im=nib.load(p);a=np.asarray(im.dataobj,dtype=np.float32).T;axis=im.header.get_axis(1)
   for h,structure in [('L','CORTEX_LEFT'),('R','CORTEX_RIGHT')]:
    v,y=extract_structure_data(axis,a,structure)
    if not np.array_equal(v,vertex_ids[h]):raise ValueError(f'Analysis/baseline brain-model mismatch: {p}')
    den[h].append(y)
  for h in ['L','R']:
   for domain,samples in [('hipp',hippocampus),('cortex',cortex)]:
    scores,new=runwise_tsnr([samples[r][h] for r in runs],frames=frames,threshold=25)
    if domain=='hipp':
     raw=c['root']/f'surface/sub-{sub}/surface/raw/sub-{sub}_hemi-{h}_space-corobl_den-512_label-hipp_bold.func.gii'
     y=np.column_stack([a.data for a in nib.load(raw).darrays])
     old=np.load(c['root']/f'surface/sub-{sub}/surface/tsnr_gate/sub-{sub}_hemi-{h}_valid_mask.npy')
    else:
     y=np.concatenate(den[h],axis=1)
     old=np.load(c['root']/f'routea/sub-{sub}/reference/common/tsnr_gate/cortex_{"left" if h=="L" else "right"}_valid_mask.npy')
    if y.shape[1]!=sum(len(f) for f in frames):raise ValueError(f'Analysis retained-frame mismatch: {branch}/{sub}/{domain}')
    new &= rest_signal_validity(y)
    np.savez(dest/f'{branch}_{domain}_{h}_qc.npz',tsnr=scores,valid=new,old_valid=old)
    rows.append(dict(dataset=ds,branch=branch,subject=sub,domain=domain,hemi=h,total=len(new),old_valid=int(old.sum()),new_valid=int(new.sum()),changed=int((new!=old).sum()),rescued=int((new&~old).sum()),newly_excluded=int((old&~new).sum()),invalid_fraction=float(1-new.mean()),run_ids=runs,retained_frames=[len(f) for f in frames]))
 return rows
rows=[];errors=[]
# Four subject workers + at most four one-thread Workbench subprocesses + driver: <=9.
with ThreadPoolExecutor(max_workers=4) as pool:
 futures={pool.submit(calculate,k,b):k for k,b in jobs.items()}
 for i,f in enumerate(as_completed(futures),1):
  key=futures[f]
  try:rows.extend(f.result());print('DONE',i,len(jobs),key,flush=True)
  except Exception as e:errors.append(dict(dataset=key[0],subject=key[1],error=str(e)));print('ERROR',i,key,str(e),flush=True)
  (OUT/'results.json').write_text(json.dumps(dict(rows=rows,errors=errors,total_subjects=len(jobs)),indent=2))
print('COMPLETE rows',len(rows),'errors',len(errors),flush=True)


{
  "rows": [
    {
      "dataset": "iowa_jj_tms",
      "branch": "jj22",
      "subject": "1001",
      "domain": "cortex",
      "hemi": "L",
      "total": 29696,
      "old_valid": 29455,
      "new_valid": 29200,
      "changed": 255,
      "rescued": 0,
      "newly_excluded": 255,
      "invalid_fraction": 0.016702586206896575,
      "run_ids": [
        1,
        2,
        3
      ],
      "retained_frames": [
        230,
        228,
        228
      ]
    },
    {
      "dataset": "iowa_jj_tms",
      "branch": "jj22",
      "subject": "1001",
      "domain": "cortex",
      "hemi": "R",
      "total": 29716,
      "old_valid": 29496,
      "new_valid": 29198,
      "changed": 298,
      "rescued": 0,
      "newly_excluded": 298,
      "invalid_fraction": 0.017431686633463506,
      "run_ids": [
        1,
        2,
        3
      ],
      "retained_frames": [
        230,
        228,
        228
      ]
    },
    {
      "dataset": "iowa_jj_tms",
      "branch": "j

### Stanford sub-4117 hippocampal supplement

This separately executed calculation supplies only the two 36P hippocampal rows omitted after the primary run could not read sub-4117’s existing cortex reference summary. The result remains a QC-mask measurement; sub-4117 already failed and still fails the hippocampal 20% rule.


In [ ]:
import sys,json,csv
from pathlib import Path
import numpy as np,nibabel as nib
sys.path.insert(0,str(Path.cwd()/'scripts/core_workflow'))
from lib.workbench import run_workbench
from lib.tsnr import runwise_tsnr
from lib.task_quality import rest_signal_validity
sub='4117';base=Path('/home/yojiao/LSS/jianglab/3_Data_Working/fmriprep_processed_CausalConnectome/rest_fmriprep');root=Path('outputs_stanford/inputs/stanford/kong2022/shared_inputs');prefix=f'sub-{sub}_task-rest';q=base/f'xcp_d_36P_Head05/sub-{sub}/func'
qc=list(csv.DictReader((q/f'{prefix}_space-MNI152NLin6Asym_desc-linc_qc.tsv').open(),delimiter='\t'))[0];mask=list(csv.DictReader((q/f'{prefix}_outliers.tsv').open(),delimiter='\t'));dummy=int(qc['num_dummy_volumes']);keep=np.array([i+dummy for i,r in enumerate(mask) if float(r['framewise_displacement'])==0],dtype=int)
rows=[]
for h in ['L','R']:
 surf=Path(f'outputs_stanford/inputs/stanford/hippunfold/dense_corobl_batch/sub-{sub}/hippunfold/sub-{sub}/surf/sub-{sub}_hemi-{h}_space-corobl_den-512_label-hipp_midthickness.surf.gii');bold=base/f'fmriprep/sub-{sub}/func/{prefix}_space-MNI152NLin6Asym_res-2_desc-preproc_bold.nii.gz';out=Path(f'/tmp/stanford_4117_{h}_baseline.func.gii')
 run_workbench(['-volume-to-surface-mapping',str(bold),str(surf),str(out),'-trilinear'])
 y=np.column_stack([a.data for a in nib.load(out).darrays]);scores,new=runwise_tsnr([y],frames=[keep],threshold=25)
 old=np.load(root/f'surface/sub-{sub}/surface/tsnr_gate/sub-{sub}_hemi-{h}_valid_mask.npy');raw=np.column_stack([a.data for a in nib.load(root/f'surface/sub-{sub}/surface/raw/sub-{sub}_hemi-{h}_space-corobl_den-512_label-hipp_bold.func.gii').darrays])
 if raw.shape[1]!=len(keep) or len(keep)!=int(qc['num_retained_volumes']):raise ValueError('Frame mismatch')
 new &= rest_signal_validity(raw)
 rows.append(dict(dataset='stanford',branch='stanford36',subject=sub,domain='hipp',hemi=h,total=512,old_valid=int(old.sum()),new_valid=int(new.sum()),changed=int((old!=new).sum()),rescued=int((new&~old).sum()),newly_excluded=int((old&~new).sum()),invalid_fraction=float(1-new.mean()),run_ids=[0],retained_frames=[len(keep)]))
 print(rows[-1],flush=True)
Path('/tmp/stanford_4117_tsnr_results.json').write_text(json.dumps(rows,indent=2))


[
  {
    "dataset": "stanford",
    "branch": "stanford36",
    "subject": "4117",
    "domain": "hipp",
    "hemi": "L",
    "total": 512,
    "old_valid": 374,
    "new_valid": 341,
    "changed": 33,
    "rescued": 0,
    "newly_excluded": 33,
    "invalid_fraction": 0.333984375,
    "run_ids": [
      0
    ],
    "retained_frames": [
      234
    ]
  },
  {
    "dataset": "stanford",
    "branch": "stanford36",
    "subject": "4117",
    "domain": "hipp",
    "hemi": "R",
    "total": 512,
    "old_valid": 387,
    "new_valid": 361,
    "changed": 26,
    "rescued": 0,
    "newly_excluded": 26,
    "invalid_fraction": 0.294921875,
    "run_ids": [
      0
    ],
    "retained_frames": [
      234
    ]
  }
]

## Cortex-mask FC impact pilot

The following executed source and JSON compare old and proposed cortical ROI means and their FC with the saved hippocampal 2/4mm time series fixed. This isolates the cortex-mask change for seven examples; it does not include changed hippocampal smoothing. The script writes only its result JSON under `/tmp`.


In [ ]:
from __future__ import annotations

import csv
import json
import warnings
from pathlib import Path

import nibabel as nib
import numpy as np

REPO = Path("/home/yojiao/hpchome/hipp")
CASES = [
    ("stanford", "1017", "36P", "stanford36"),
    ("stanford", "1039", "36P", "stanford36"),
    ("stanford", "1046", "36P", "stanford36"),
    ("stanford", "4078", "36P", "stanford36"),
    ("stanford", "1039", "22P", "stanford22"),
    ("iowa_jj_tms", "1001", "36P", "jj36"),
    ("iowa_jj_tms", "1001", "22P", "jj22"),
]
SMOOTHINGS = ("2mm", "4mm")


def case_base(dataset: str, branch: str) -> Path:
    if dataset == "stanford":
        tree = "kong2022" if branch == "36P" else "kong2022_22p"
        return REPO / f"outputs_stanford/inputs/stanford/{tree}/shared_inputs"
    tree = "kong2022/shared_inputs" if branch == "36P" else "kong2022/shared_inputs_22p"
    return REPO / f"outputs_iowa_jj_tms/inputs/iowa_jj_tms/{tree}"


def load_source(case: dict) -> tuple[dict[str, np.ndarray], dict[str, np.ndarray], int]:
    dataset, subject, branch = case["id"].split("/")
    ref = case["ref"]
    expected_lengths = None
    if dataset == "stanford":
        summary = json.loads((ref / "reference_summary.json").read_text())
        source_paths = [Path(summary["dtseries"])]
        label_paths = {"L": Path(summary["route_a_lh_labels"]), "R": Path(summary["route_a_rh_labels"])}
    elif branch == "22P":
        summary = json.loads((ref / "reference_summary.json").read_text())
        source_paths = [Path(p) for p in summary["dtseries_inputs"]]
        label_paths = {"L": Path(summary["route_a_lh_labels"]), "R": Path(summary["route_a_rh_labels"])}
        expected_lengths = [int(n) for n in summary["source_run_lengths"]]
        if len(source_paths) != len(expected_lengths):
            raise ValueError("JJ22 dtseries_inputs and source_run_lengths differ in length")
    else:
        manifest_path = REPO / f"outputs_iowa_jj_tms/work/hcp151_fixed_template_matching/subjects/sub-{subject}/subject_manifest.json"
        manifest = json.loads(manifest_path.read_text())
        inputs = manifest["inputs"]
        run_evidence = manifest["header_evidence"]["runs"]
        run_ids = [int(run["run_id"]) for run in run_evidence]
        if len(set(run_ids)) != len(run_ids):
            raise ValueError("JJ36 manifest has duplicate run IDs")
        source_paths = []
        expected_lengths = []
        for run, run_id in zip(run_evidence, run_ids):
            item = inputs[f"run-{run_id:02d}_dtseries"]
            source_paths.append(Path(item["path"]))
            expected_lengths.append(int(run["dtseries_timepoints"]))
        label_paths = {
            "L": Path(inputs["areal_lh_labels"]["path"]),
            "R": Path(inputs["areal_rh_labels"]["path"]),
        }

    chunks: dict[str, list[np.ndarray]] = {"L": [], "R": []}
    vertex_maps: dict[str, np.ndarray] = {}
    observed_lengths = []
    for run_index, path in enumerate(source_paths):
        if not path.is_file():
            raise FileNotFoundError(f"Declared dtseries is missing: {path}")
        image = nib.load(str(path))
        observed_lengths.append(int(image.shape[0]))
        if expected_lengths is not None and observed_lengths[-1] != expected_lengths[run_index]:
            raise ValueError(
                f"Declared run length mismatch for {path}: "
                f"manifest={expected_lengths[run_index]}, CIFTI={observed_lengths[-1]}"
            )
        brain_axis = image.header.get_axis(1)
        found = set()
        for structure, gray_slice, model in brain_axis.iter_structures():
            if structure == "CIFTI_STRUCTURE_CORTEX_LEFT":
                hemi = "L"
            elif structure == "CIFTI_STRUCTURE_CORTEX_RIGHT":
                hemi = "R"
            else:
                continue
            found.add(hemi)
            vertices = np.asarray(model.vertex, dtype=np.int64)
            if hemi in vertex_maps and not np.array_equal(vertex_maps[hemi], vertices):
                raise ValueError(f"Cortex vertex mapping changed across dtseries runs for hemi-{hemi}")
            vertex_maps[hemi] = vertices
            block = np.asarray(image.dataobj[:, gray_slice], dtype=np.float32).T
            if block.shape[0] != vertices.size:
                raise ValueError(f"CIFTI cortex block and vertex map differ for hemi-{hemi}")
            chunks[hemi].append(block)
        if found != {"L", "R"}:
            raise ValueError(f"Declared CIFTI is missing a cortex structure: {path}")

    if expected_lengths is not None and len(observed_lengths) != len(expected_lengths):
        raise ValueError("Loaded run count differs from the declared run count")
    data = {hemi: np.concatenate(chunks[hemi], axis=1) for hemi in ("L", "R")}
    labels = {}
    for hemi in ("L", "R"):
        full = np.loadtxt(label_paths[hemi], dtype=np.int64)
        if full.ndim != 1 or vertex_maps[hemi].size and int(vertex_maps[hemi].max()) >= full.size:
            raise ValueError(f"Label vector cannot index the CIFTI surface vertices for hemi-{hemi}")
        labels[hemi] = full[vertex_maps[hemi]]
    return data, labels, int(sum(observed_lengths))


def summarize(values: list[float] | np.ndarray) -> dict:
    arr = np.asarray(values, dtype=np.float64).reshape(-1)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return {"n": 0, "min": None, "median": None, "p05": None, "p95": None, "max": None}
    return {
        "n": int(arr.size),
        "min": float(np.min(arr)),
        "median": float(np.median(arr)),
        "p05": float(np.percentile(arr, 5)),
        "p95": float(np.percentile(arr, 95)),
        "max": float(np.max(arr)),
    }


def pearson_series(a: np.ndarray, b: np.ndarray) -> float:
    a = np.asarray(a, dtype=np.float64).reshape(-1)
    b = np.asarray(b, dtype=np.float64).reshape(-1)
    good = np.isfinite(a) & np.isfinite(b)
    if int(good.sum()) < 2:
        return float("nan")
    x = a[good] - np.mean(a[good])
    y = b[good] - np.mean(b[good])
    denom = np.linalg.norm(x) * np.linalg.norm(y)
    return float(np.dot(x, y) / denom) if denom > 0 else float("nan")


def corr_vertices_to_rois(vertices: np.ndarray, rois: np.ndarray) -> np.ndarray:
    x = np.asarray(vertices, dtype=np.float64)
    y = np.asarray(rois, dtype=np.float64)
    if x.ndim != 2 or y.ndim != 2 or x.shape[1] != y.shape[1]:
        raise ValueError("Hippocampal and ROI time-series dimensions do not align")
    if np.isfinite(x).all() and np.isfinite(y).all():
        xc = x - x.mean(axis=1, keepdims=True)
        yc = y - y.mean(axis=1, keepdims=True)
        denom = np.linalg.norm(xc, axis=1)[:, None] * np.linalg.norm(yc, axis=1)[None, :]
        out = np.full((x.shape[0], y.shape[0]), np.nan, dtype=np.float64)
        good = denom > 0
        out[good] = (xc @ yc.T)[good] / denom[good]
        return out
    out = np.full((x.shape[0], y.shape[0]), np.nan, dtype=np.float64)
    for index, roi in enumerate(y):
        paired = np.isfinite(x) & np.isfinite(roi)[None, :]
        counts = paired.sum(axis=1)
        x0 = np.where(paired, x, 0.0)
        y0 = np.where(paired, roi[None, :], 0.0)
        with np.errstate(invalid="ignore", divide="ignore"):
            mx = x0.sum(axis=1) / counts
            my = y0.sum(axis=1) / counts
        dx = np.where(paired, x - mx[:, None], 0.0)
        dy = np.where(paired, roi[None, :] - my[:, None], 0.0)
        denom = np.sqrt((dx * dx).sum(axis=1) * (dy * dy).sum(axis=1))
        good = (counts >= 2) & (denom > 0)
        out[good, index] = (dx[good] * dy[good]).sum(axis=1) / denom[good]
    return out


def run_case(dataset: str, subject: str, branch: str, qc_tag: str) -> dict:
    case_id = f"{dataset}/{subject}/{branch}"
    base = case_base(dataset, branch)
    ref = base / f"routea/sub-{subject}/reference/kong2022"
    qc_dir = Path("/tmp/full_rest_tsnr") / dataset / subject
    if not ref.is_dir():
        raise FileNotFoundError(f"Formal reference directory is missing: {ref}")
    if not qc_dir.is_dir():
        raise FileNotFoundError(f"New tSNR QC directory is missing: {qc_dir}")

    data, label_by_hemi, n_time = load_source({"id": case_id, "ref": ref})
    table_path = ref / "cortex_roi_parcels.tsv"
    stored_path = ref / "cortex_roi_parcel_timeseries.npy"
    with table_path.open(newline="") as stream:
        rows = list(csv.DictReader(stream, delimiter="\t"))
    stored = np.load(stored_path)
    if stored.ndim != 2 or stored.shape[0] != len(rows) or stored.shape[1] != n_time:
        raise ValueError(
            f"Stored ROI array shape {stored.shape} does not align with {len(rows)} rows and {n_time} timepoints"
        )

    metadata: dict[tuple[str, int], int] = {}
    for row_index, row in enumerate(rows):
        key = (row["hemisphere"], int(row["parcel_id"]))
        if key in metadata:
            raise ValueError(f"Duplicate parcel metadata row: {key}")
        metadata[key] = row_index

    old_masks = {}
    new_masks = {}
    changed_vertices = {}
    changed_parcel_keys: set[tuple[str, int]] = set()
    newly_empty_keys = set()
    old_ts: dict[tuple[str, int], np.ndarray] = {}
    new_ts: dict[tuple[str, int], np.ndarray] = {}
    old_counts = {}
    new_counts = {}

    for hemi, side in (("L", "left"), ("R", "right")):
        qpath = qc_dir / f"{qc_tag}_cortex_{hemi}_qc.npz"
        with np.load(qpath) as qc:
            new_mask = np.asarray(qc["valid"], dtype=bool).reshape(-1)
            qc_old = np.asarray(qc["old_valid"], dtype=bool).reshape(-1)
        official_path = ref.parent / "common/tsnr_gate" / f"cortex_{side}_valid_mask.npy"
        official_old = np.asarray(np.load(official_path), dtype=bool).reshape(-1)
        if not np.array_equal(qc_old, official_old):
            raise ValueError(f"QC old_valid differs from the formal original mask: {official_path}")
        if new_mask.size != data[hemi].shape[0] or official_old.size != data[hemi].shape[0]:
            raise ValueError(f"Mask and CIFTI vertex counts differ for hemi-{hemi}")
        labels = label_by_hemi[hemi]
        if labels.size != data[hemi].shape[0]:
            raise ValueError(f"Label and CIFTI vertex counts differ for hemi-{hemi}")
        old_masks[hemi] = official_old
        new_masks[hemi] = new_mask
        changed = official_old != new_mask
        changed_vertices[hemi] = {
            "old_valid": int(official_old.sum()),
            "new_valid": int(new_mask.sum()),
            "added": int(np.count_nonzero(~official_old & new_mask)),
            "removed": int(np.count_nonzero(official_old & ~new_mask)),
            "changed": int(changed.sum()),
        }

        pids = sorted(int(pid) for pid in np.unique(labels) if int(pid) > 0)
        expected_old_keys = set()
        for pid in pids:
            membership_old = official_old & (labels == pid)
            membership_new = new_mask & (labels == pid)
            n_old = int(membership_old.sum())
            n_new = int(membership_new.sum())
            key = (hemi, pid)
            old_counts[key] = n_old
            new_counts[key] = n_new
            if n_old:
                expected_old_keys.add(key)
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore", RuntimeWarning)
                    old_ts[key] = np.nanmean(data[hemi][membership_old, :], axis=0).astype(np.float32)
            if n_new:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore", RuntimeWarning)
                    new_ts[key] = np.nanmean(data[hemi][membership_new, :], axis=0).astype(np.float32)
            if not np.array_equal(membership_old, membership_new):
                changed_parcel_keys.add(key)
            if n_old > 0 and n_new == 0:
                newly_empty_keys.add(key)

        metadata_keys = {key for key in metadata if key[0] == hemi}
        if metadata_keys != expected_old_keys:
            missing = sorted(expected_old_keys - metadata_keys)
            extra = sorted(metadata_keys - expected_old_keys)
            raise ValueError(
                f"Original metadata does not cover old-mask parcels for hemi-{hemi}; "
                f"missing={missing[:8]}, extra={extra[:8]}"
            )

    for key, row_index in metadata.items():
        reconstructed = old_ts[key]
        np.testing.assert_allclose(
            reconstructed,
            stored[row_index],
            rtol=1e-4,
            atol=1e-4,
            equal_nan=True,
            err_msg=f"Original mask reconstruction differs from stored ROI time series for {key}",
        )

    common_keys = sorted(set(old_ts) & set(new_ts))
    if not common_keys:
        raise ValueError("No nonempty ROI parcels are common to old and new masks")
    changed_common_keys = [key for key in common_keys if key in changed_parcel_keys]
    roi_corrs = [
        pearson_series(old_ts[key], new_ts[key])
        for key in changed_common_keys
    ]

    all_fc_deltas = []
    changed_fc_deltas = []
    fc_by_setting = {}
    for smoothing in SMOOTHINGS:
        for hipp_hemi in ("L", "R"):
            hipp_path = (
                base
                / f"surface/sub-{subject}/surface/{smoothing}/sub-{subject}_hemi-{hipp_hemi}_timeseries.npy"
            )
            hipp = np.asarray(np.load(hipp_path), dtype=np.float32)
            if hipp.ndim != 2 or hipp.shape[1] != n_time:
                raise ValueError(
                    f"Fixed hippocampal time series {hipp_path} has shape {hipp.shape}; expected time length {n_time}"
                )
            old_matrix = np.stack([old_ts[key] for key in common_keys])
            new_matrix = np.stack([new_ts[key] for key in common_keys])
            old_fc = corr_vertices_to_rois(hipp, old_matrix)
            new_fc = corr_vertices_to_rois(hipp, new_matrix)
            finite = np.isfinite(old_fc) & np.isfinite(new_fc)
            delta = np.abs(new_fc - old_fc)
            all_values = delta[finite]
            all_fc_deltas.append(all_values)
            changed_columns = [index for index, key in enumerate(common_keys) if key in changed_parcel_keys]
            if changed_columns:
                changed_delta = delta[:, changed_columns]
                changed_finite = finite[:, changed_columns]
                changed_values = changed_delta[changed_finite]
            else:
                changed_values = np.empty(0, dtype=np.float64)
            changed_fc_deltas.append(changed_values)
            fc_by_setting[f"{smoothing}_hipp-{hipp_hemi}"] = {
                "all_common_finite": summarize(all_values),
                "changed_parcel_common_finite": summarize(changed_values),
            }

    all_values = np.concatenate(all_fc_deltas) if all_fc_deltas else np.empty(0)
    changed_values = np.concatenate(changed_fc_deltas) if changed_fc_deltas else np.empty(0)
    return {
        "status": "ok",
        "case": case_id,
        "timepoints": n_time,
        "fixed_hippocampal_timeseries": True,
        "changed_parcel_count": int(len(changed_parcel_keys)),
        "changed_parcels_by_hemi": {
            hemi: int(sum(key[0] == hemi for key in changed_parcel_keys))
            for hemi in ("L", "R")
        },
        "newly_empty_parcel_count": int(len(newly_empty_keys)),
        "newly_empty_parcels_by_hemi": {
            hemi: int(sum(key[0] == hemi for key in newly_empty_keys))
            for hemi in ("L", "R")
        },
        "newly_nonempty_parcel_count": int(
            sum(old_counts.get(key, 0) == 0 and new_counts.get(key, 0) > 0 for key in new_counts)
        ),
        "mask_vertices_by_hemi": changed_vertices,
        "roi_timeseries_correlation_changed_parcels": summarize(roi_corrs),
        "abs_delta_fc_all_jointly_finite": summarize(all_values),
        "abs_delta_fc_changed_parcels_jointly_finite": summarize(changed_values),
        "fc_by_hipp_surface": fc_by_setting,
    }


def main() -> None:
    results = []
    for dataset, subject, branch, qc_tag in CASES:
        case_id = f"{dataset}/{subject}/{branch}"
        print(f"START {case_id}", flush=True)
        try:
            result = run_case(dataset, subject, branch, qc_tag)
        except Exception as exc:
            result = {"status": "failed", "case": case_id, "error": f"{type(exc).__name__}: {exc}"}
        results.append(result)
        print(json.dumps(result, separators=(",", ":"), allow_nan=False), flush=True)
    output = {
        "analysis": "direct rest FC effect of new cortex tSNR masks with fixed original hippocampal timeseries",
        "cases": results,
        "method_limits": [
            "Hippocampal 2mm and 4mm time series were held fixed; this does not include changes from recomputing hippocampal smoothing or a full cortex-plus-hippocampus validity merge.",
            "FC deltas compare old-mask and new-mask cortical ROI means against the saved hippocampal vertex time series; only jointly finite FC elements enter the reported distributions.",
            "Original ROI means were reconstructed from the formal old masks and checked against the stored ROI time series before any delta was calculated.",
        ],
    }
    out_path = Path("/tmp/rest_cortex_fc_effect.json")
    out_path.write_text(json.dumps(output, indent=2, allow_nan=False) + "\n")
    print(f"WROTE {out_path}", flush=True)


if __name__ == "__main__":
    main()


{
  "analysis": "direct rest FC effect of new cortex tSNR masks with fixed original hippocampal timeseries",
  "cases": [
    {
      "status": "ok",
      "case": "stanford/1017/36P",
      "timepoints": 231,
      "fixed_hippocampal_timeseries": true,
      "changed_parcel_count": 80,
      "changed_parcels_by_hemi": {
        "L": 30,
        "R": 50
      },
      "newly_empty_parcel_count": 0,
      "newly_empty_parcels_by_hemi": {
        "L": 0,
        "R": 0
      },
      "newly_nonempty_parcel_count": 0,
      "mask_vertices_by_hemi": {
        "L": {
          "old_valid": 29453,
          "new_valid": 29156,
          "added": 0,
          "removed": 297,
          "changed": 297
        },
        "R": {
          "old_valid": 29488,
          "new_valid": 29036,
          "added": 0,
          "removed": 452,
          "changed": 452
        }
      },
      "roi_timeseries_correlation_changed_parcels": {
        "n": 80,
        "min": 0.7881352509975863,
        "media

## Post-review tSNR threshold sensitivity

The next code cell re-evaluates saved runwise scores at 10, 15, 20, 25, and 30. It keeps the existing all-runs-pass rule, the task `<20% invalid` and Dice criteria, and rest finite/nonconstant signal validity. For rest it consumes the per-run scores and GIFTI outputs created by the full-cohort cell above. Stanford 36P sub-4117 was separately calculated and is excluded from that sweep; it already failed the historical 20% coverage rule. For HCP task, scores for the existing 25 and additional 9 subjects live under their respective explicitly named QC roots. This is a sensitivity analysis; it does not select or deploy a replacement threshold.


In [ ]:
import csv
import json
from pathlib import Path

import nibabel as nib
import numpy as np

THRESHOLDS = (10, 15, 20, 25, 30)
REPO = Path("/home/yojiao/hpchome/hipp")
CACHE = Path("/tmp/full_rest_tsnr")
rest_roots = {
    "stanford36": REPO / "outputs_stanford/inputs/stanford/kong2022/shared_inputs",
    "stanford22": REPO / "outputs_stanford/inputs/stanford/kong2022_22p/shared_inputs",
    "jj36": REPO / "outputs_iowa_jj_tms/inputs/iowa_jj_tms/kong2022/shared_inputs",
    "jj22": REPO / "outputs_iowa_jj_tms/inputs/iowa_jj_tms/kong2022/shared_inputs_22p",
}
rest_rows = json.loads((CACHE / "results.json").read_text())["rows"]
print("REST: branch, cutoff, all-run valid/total, any-run tSNR pass, subjects with either hemi >=20% invalid, mask mismatches at 25")
for branch, root in rest_roots.items():
    branch_rows = [r for r in rest_rows if r["branch"] == branch and r["domain"] == "hipp"]
    scores = {}
    validity = {}
    reference_masks = {}
    for row in branch_rows:
        subject, hemi = row["subject"], row["hemi"]
        cache = CACHE / row["dataset"] / subject / f"{branch}_hipp_{hemi}_qc.npz"
        saved = np.load(cache, allow_pickle=False)
        raw = root / f"surface/sub-{subject}/surface/raw/sub-{subject}_hemi-{hemi}_space-corobl_den-512_label-hipp_bold.func.gii"
        series = np.column_stack([a.data for a in nib.load(raw).darrays]).astype(float)
        finite = np.isfinite(series).all(axis=1)
        valid_signal = np.zeros(series.shape[0], dtype=bool)
        valid_signal[finite] = np.ptp(series[finite], axis=1) > 0
        scores.setdefault(subject, {})[hemi] = saved["tsnr"]
        validity.setdefault(subject, {})[hemi] = valid_signal
        reference_masks.setdefault(subject, {})[hemi] = saved["valid"]
    mismatch25 = 0
    for subject in scores:
        for hemi, values in scores[subject].items():
            mask25 = np.isfinite(values).all(axis=0) & (values >= 25).all(axis=0) & validity[subject][hemi]
            mismatch25 += int(np.count_nonzero(mask25 != reference_masks[subject][hemi]))
    for cutoff in THRESHOLDS:
        n_valid = n_any = n_failed = 0
        for subject in scores:
            subject_failed = False
            for hemi, values in scores[subject].items():
                per_run = np.isfinite(values) & (values >= cutoff)
                mask = per_run.all(axis=0) & validity[subject][hemi]
                n_valid += int(mask.sum())
                n_any += int(per_run.any(axis=0).sum())
                subject_failed |= (1.0 - float(mask.mean())) >= 0.20
            n_failed += int(subject_failed)
        print(f"{branch}, {cutoff}, {n_valid}/{len(branch_rows)*512}, {n_any}, {n_failed}/{len(scores)}, {mismatch25 if cutoff == 25 else ''}")

print("TASK: branch, cutoff, all-run valid/total, any-run tSNR pass, subjects passing tSNR, subjects passing tSNR + Dice")
shared = REPO / "outputs_hcp/inputs/hcp/kong2022/shared_inputs"
for task in ("emotion", "wm"):
    task_root = shared / f"task-{task}"
    rows, cohort = [], {}
    for tag in ("existing25", "additional9"):
        manifest = task_root / "qc" / tag / "t1only_hippunfold_tsnr_qc.tsv"
        members = list(csv.DictReader(manifest.open(), delimiter="\t"))
        rows.extend(members)
        for member in members:
            cohort[member["subject"]] = tag
    if len(rows) != 34 or len(cohort) != 34:
        raise ValueError(f"Expected the 34 declared {task} task subjects")
    scores, dice, mismatch25 = {}, {}, 0
    for member in rows:
        subject = member["subject"]
        dice[subject] = member["pass_dice"].lower() == "true"
        if cohort[subject] == "existing25":
            gate = task_root / "surface" / subject / "surface" / "tsnr_gate"
        else:
            gate = task_root / subject / "surface" / "tsnr_gate"
        scores[subject] = {}
        for hemi in ("L", "R"):
            values = np.load(gate / f"{subject}_hemi-{hemi}_run_tsnr.npy", allow_pickle=False)
            saved_mask = np.load(gate / f"{subject}_hemi-{hemi}_valid_mask.npy", allow_pickle=False)
            mask25 = np.isfinite(values).all(axis=0) & (values >= 25).all(axis=0)
            mismatch25 += int(np.count_nonzero(mask25 != saved_mask))
            scores[subject][hemi] = values
    for cutoff in THRESHOLDS:
        n_valid = n_any = n_tsnr = n_both = 0
        for subject in scores:
            masks = {}
            for hemi, values in scores[subject].items():
                per_run = np.isfinite(values) & (values >= cutoff)
                masks[hemi] = per_run.all(axis=0)
                n_valid += int(masks[hemi].sum())
                n_any += int(per_run.any(axis=0).sum())
            passes = all((1.0 - float(masks[hemi].mean())) < 0.20 for hemi in ("L", "R"))
            n_tsnr += int(passes)
            n_both += int(passes and dice[subject])
        print(f"{task}, {cutoff}, {n_valid}/34816, {n_any}, {n_tsnr}/34, {n_both}/34, threshold-25 mask mismatches={mismatch25 if cutoff == 25 else 'n/a'}")
